# Houston 311 Road Issue Hotspot Analysis

This notebook walks through the full project workflow: data source, loading, cleaning, filtering, aggregation, priority scoring, processed file creation, and visual outputs.

**Research question:** Which Houston ZIP-code areas show recurring flooding and pothole complaints, and which overlap areas should be prioritized for maintenance inspection?

## 1. Data Source

The raw data comes from the official City of Houston 311 Service Request Data page:

https://www.houstontx.gov/311/servicerequestdata.html

This project uses the public piped extracts for:

- `2025 (piped)`
- `2026 YTD (piped)`

The raw files are large, so they are not committed to GitHub. To reproduce this notebook, download them into `data/raw/` using these names:

```text
data/raw/houston_311_2025_piped.txt
data/raw/houston_311_2026_ytd_piped.txt
```

In [ ]:
from pathlib import Path
import csv
import json

import pandas as pd

ROOT = Path('..')
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
FIGURES_DIR = ROOT / 'figures'

RAW_FILES = [
    RAW_DIR / 'houston_311_2025_piped.txt',
    RAW_DIR / 'houston_311_2026_ytd_piped.txt',
]

USECOLS = [
    'Case Number', 'Incident Address', 'Latitude', 'Longitude', 'Status',
    'Created Date Local', 'Closed Date', 'Incident Case Type', 'Department',
    'Division', 'Zip Code', 'Customer SuperNeighborhood', 'Channel'
]

ISSUE_TYPES = {'Flooding': 'Flooding', 'Pothole': 'Pothole'}
STUDY_START = pd.Timestamp('2025-01-01')

## 2. Load and Filter Raw 311 Extracts

The source files are pipe-delimited text files with a few metadata rows before the header. The notebook reads only the columns needed for the analysis and filters the records to the exact case types `Flooding` and `Pothole`.

In [ ]:
frames = []
raw_rows = 0

for path in RAW_FILES:
    if not path.exists():
        raise FileNotFoundError(f'Missing raw file: {path}')

    chunks = pd.read_csv(
        path,
        sep='|',
        skiprows=5,
        usecols=USECOLS,
        dtype=str,
        encoding='latin1',
        engine='python',
        quoting=csv.QUOTE_NONE,
        on_bad_lines='skip',
        chunksize=75_000,
    )

    for chunk in chunks:
        raw_rows += len(chunk)
        mask = chunk['Incident Case Type'].isin(ISSUE_TYPES)
        frames.append(chunk.loc[mask].copy())

raw_filtered = pd.concat(frames, ignore_index=True)
raw_filtered.shape, raw_rows

In [ ]:
raw_filtered[['Case Number', 'Created Date Local', 'Incident Case Type', 'Zip Code', 'Latitude', 'Longitude']].head()

## 3. Clean and Organize Data

Cleaning steps:

- convert created date to datetime
- keep records from 2025 onward
- standardize ZIP codes to five-digit values
- convert latitude and longitude to numeric fields
- remove duplicate case numbers
- flag coordinates that fall inside a broad Houston-area bounding box

In [ ]:
df = raw_filtered.copy()
df['issue_type'] = df['Incident Case Type'].map(ISSUE_TYPES)
df['created_at'] = pd.to_datetime(df['Created Date Local'], errors='coerce')
df = df.dropna(subset=['created_at'])
df = df[df['created_at'] >= STUDY_START].copy()

df['year'] = df['created_at'].dt.year
df['month'] = df['created_at'].dt.to_period('M').astype(str)
df['month_name'] = df['created_at'].dt.strftime('%b %Y')
df['latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df['zip_code'] = (
    df['Zip Code'].fillna('').astype(str)
    .str.extract(r'(\d{5})', expand=False)
    .fillna('Unknown')
)

before_dedupe = len(df)
df = df.drop_duplicates(subset=['Case Number'], keep='first')
duplicates_removed = before_dedupe - len(df)

df['valid_coordinate'] = (
    df['latitude'].between(29.0, 30.3) &
    df['longitude'].between(-96.0, -94.5)
)

df[['Case Number', 'issue_type', 'created_at', 'month', 'zip_code', 'latitude', 'longitude', 'valid_coordinate']].head()

In [ ]:
summary_metrics = {
    'raw_records_loaded': raw_rows,
    'filtered_records': len(df),
    'duplicates_removed': duplicates_removed,
    'flooding_complaints': int((df['issue_type'] == 'Flooding').sum()),
    'pothole_complaints': int((df['issue_type'] == 'Pothole').sum()),
    'valid_mapped_records': int(df['valid_coordinate'].sum()),
    'date_min': df['created_at'].min().strftime('%Y-%m-%d'),
    'date_max': df['created_at'].max().strftime('%Y-%m-%d'),
}
summary_metrics

## 4. Complaint Totals and Monthly Trends

In [ ]:
totals = (
    df.groupby('issue_type')
    .size()
    .rename('complaints')
    .reset_index()
    .sort_values('complaints', ascending=False)
)
totals

In [ ]:
monthly = (
    df.groupby(['month', 'issue_type'])
    .size()
    .rename('complaints')
    .reset_index()
    .sort_values(['month', 'issue_type'])
)
monthly.head(10)

In [ ]:
peak_by_type = {}
for issue_type, group in monthly.groupby('issue_type'):
    top = group.sort_values('complaints', ascending=False).iloc[0]
    peak_by_type[issue_type] = {'month': top['month'], 'complaints': int(top['complaints'])}

peak_by_type

## 5. ZIP-Code Hotspot Scoring

The priority score is designed as a screening signal, not a final engineering decision rule.

```text
Priority Score = Pothole Count + 2 x Flooding Count + 10 x Overlap Flag
```

Flooding is weighted higher because drainage-related issues can compound pavement damage and safety risk. The overlap bonus highlights ZIP codes where both complaint types appear during the study window.

In [ ]:
zip_summary = (
    df[df['zip_code'] != 'Unknown']
    .pivot_table(
        index='zip_code',
        columns='issue_type',
        values='Case Number',
        aggfunc='count',
        fill_value=0,
    )
    .reset_index()
)

for col in ['Flooding', 'Pothole']:
    if col not in zip_summary.columns:
        zip_summary[col] = 0

zip_summary['total_complaints'] = zip_summary['Flooding'] + zip_summary['Pothole']
zip_summary['overlap_flag'] = ((zip_summary['Flooding'] > 0) & (zip_summary['Pothole'] > 0)).astype(int)
zip_summary['priority_score'] = 2 * zip_summary['Flooding'] + zip_summary['Pothole'] + 10 * zip_summary['overlap_flag']
zip_summary = zip_summary.sort_values('priority_score', ascending=False)

summary_metrics['zip_codes_with_overlap'] = int(zip_summary['overlap_flag'].sum())
summary_metrics['top_priority_zip_codes'] = zip_summary.head(5)['zip_code'].tolist()
summary_metrics['peak_by_type'] = peak_by_type

zip_summary.head(10)

## 6. Save Processed Outputs

The GitHub repo stores summary-level processed files instead of full address-level case records. This keeps the repo lightweight and avoids republishing unnecessary incident-address detail.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

sample_cols = ['issue_type', 'month', 'zip_code', 'latitude', 'longitude', 'valid_coordinate']
mapped_sample = (
    df[df['valid_coordinate']]
    .loc[:, sample_cols]
    .sample(min(500, int(df['valid_coordinate'].sum())), random_state=11)
    .sort_values(['issue_type', 'zip_code'])
)

totals.to_csv(PROCESSED_DIR / 'complaint_totals.csv', index=False)
monthly.to_csv(PROCESSED_DIR / 'monthly_complaint_trends.csv', index=False)
zip_summary.to_csv(PROCESSED_DIR / 'hotspot_summary_by_zip.csv', index=False)
mapped_sample.to_csv(PROCESSED_DIR / 'mapped_cases_sample.csv', index=False)
(PROCESSED_DIR / 'metrics.json').write_text(json.dumps(summary_metrics, indent=2), encoding='utf-8')

list(PROCESSED_DIR.glob('*'))

## 7. Visual Outputs

The final visuals are generated by `src/analyze_311.py` and saved as SVG files so they render cleanly in GitHub README pages.

The visuals include:

- KPI summary
- complaint totals
- monthly trend
- priority score explanation
- top priority ZIP codes
- spatial complaint distribution
- updated poster

In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(FIGURES_DIR / 'kpi_summary.svg')))
display(SVG(filename=str(FIGURES_DIR / 'complaint_totals.svg')))
display(SVG(filename=str(FIGURES_DIR / 'monthly_trend.svg')))
display(SVG(filename=str(FIGURES_DIR / 'top_priority_zip_codes.svg')))
display(SVG(filename=str(FIGURES_DIR / 'complaint_map.svg')))

## 8. Final Findings

- Pothole complaints greatly outnumber flooding complaints in the filtered dataset.
- 77 ZIP codes had both flooding and pothole complaints during the study window.
- The highest-priority ZIP codes in the rebuilt analysis were 77007, 77008, 77082, 77006, and 77009.
- The analysis should be treated as a screening tool. Field inspection, road segment data, maintenance history, traffic volume, and drainage infrastructure data would be needed before operational decisions.